In [45]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

from src.schema import DATE_COL, STORE_COL, ITEM_COL, SALES_COL
from src.data import load_data, clean_data, time_split
from src.metrics import wape, rmse, mape, bias

from src.backtest import rolling_origin_windows, evaluate_forecast, run_backtest

print("All imports OK")

All imports OK


In [46]:
df_raw = pd.read_csv('../data/raw/train.csv')
df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL])
df = df_raw.sort_values([STORE_COL, ITEM_COL, DATE_COL]).reset_index(drop=True)
print("Loaded:", df.shape)
print("Date range:", df[DATE_COL].min(), "→", df[DATE_COL].max())

Loaded: (913000, 4)
Date range: 2013-01-01 00:00:00 → 2017-12-31 00:00:00


In [47]:
def rolling_origin_windows(df, horizon=28, n_windows=3):
    """
    Generates n_windows (train, val) splits.
    Each val window is 'horizon' days.
    Windows roll forward — val of window i becomes part of train in window i+1.
    """
    all_dates = sorted(df[DATE_COL].unique())
    total_days = len(all_dates)
    
    windows = []
    for i in range(n_windows):
        # Val window ends at: last date, then second-last 28-day block, etc.
        val_end_idx   = total_days - 1 - (i * horizon)
        val_start_idx = val_end_idx - horizon + 1
        
        val_end   = all_dates[val_end_idx]
        val_start = all_dates[val_start_idx]
        
        train_df = df[df[DATE_COL] < val_start].copy()
        val_df   = df[(df[DATE_COL] >= val_start) & (df[DATE_COL] <= val_end)].copy()
        
        windows.append({
            'window': i + 1,
            'train': train_df,
            'val': val_df,
            'val_start': val_start,
            'val_end': val_end
        })
    
    # Return in chronological order (window 1 = earliest)
    windows.reverse()
    return windows

windows = rolling_origin_windows(df, horizon=28, n_windows=3)
for w in windows:
    print(f"Window {w['window']}: val {w['val_start'].date()} → {w['val_end'].date()} | train rows: {len(w['train'])} | val rows: {len(w['val'])}")

Window 3: val 2017-10-09 → 2017-11-05 | train rows: 871000 | val rows: 14000
Window 2: val 2017-11-06 → 2017-12-03 | train rows: 885000 | val rows: 14000
Window 1: val 2017-12-04 → 2017-12-31 | train rows: 899000 | val rows: 14000


In [48]:
def naive_forecast(train_df, val_df):
    """
    Naive: predict each (store, item)'s sales on day t 
    as its sales exactly 28 days ago in train.
    Output format: date, store, item, y_hat, model
    """
    last_28 = train_df[train_df[DATE_COL] >= train_df[DATE_COL].max() - pd.Timedelta(days=27)].copy()
    
    # Map each val date to 28 days earlier
    val_copy = val_df[[DATE_COL, STORE_COL, ITEM_COL, SALES_COL]].copy()
    val_copy['lookup_date'] = val_copy[DATE_COL] - pd.Timedelta(days=28)
    
    lookup = last_28[[DATE_COL, STORE_COL, ITEM_COL, SALES_COL]].rename(
        columns={DATE_COL: 'lookup_date', SALES_COL: 'y_hat'}
    )
    
    result = val_copy.merge(lookup, on=['lookup_date', STORE_COL, ITEM_COL], how='left')
    train_mean = train_df[SALES_COL].mean()
    result['y_hat'] = result['y_hat'].fillna(train_mean)
    result['model'] = 'naive'
    
    return result[[DATE_COL, STORE_COL, ITEM_COL, 'y_hat', 'model']]

# Test on window 1
sample_forecast = naive_forecast(windows[0]['train'], windows[0]['val'])
print(sample_forecast.head())
print("Shape:", sample_forecast.shape)

        date  store  item  y_hat  model
0 2017-10-09      1     1     16  naive
1 2017-10-10      1     1     20  naive
2 2017-10-11      1     1     13  naive
3 2017-10-12      1     1     26  naive
4 2017-10-13      1     1     28  naive
Shape: (14000, 5)


In [49]:
def evaluate_forecast(forecast_df, val_df, window_num):
    """
    Takes a forecast df (date, store, item, y_hat, model)
    and val df (date, store, item, sales).
    Returns a row per model with wape, rmse, mape, bias scores.
    """
    merged = val_df[[DATE_COL, STORE_COL, ITEM_COL, SALES_COL]].merge(
        forecast_df[[DATE_COL, STORE_COL, ITEM_COL, 'y_hat', 'model']],
        on=[DATE_COL, STORE_COL, ITEM_COL],
        how='inner'
    )
    
    results = []
    for model_name, group in merged.groupby('model'):
        results.append({
            'window':  window_num,
            'model':   model_name,
            'wape':    wape(group[SALES_COL], group['y_hat']),
            'rmse':    rmse(group[SALES_COL], group['y_hat']),
            'mape':    mape(group[SALES_COL], group['y_hat']),
            'bias':    bias(group[SALES_COL], group['y_hat']),
            'n_rows':  len(group)
        })
    return pd.DataFrame(results)

In [50]:
def run_backtest(forecast_fn, df, n_windows=3, horizon=28):
    """
    Runs forecast_fn on each rolling window and aggregates metrics.
    forecast_fn(train_df, val_df) -> forecast_df
    """
    windows = rolling_origin_windows(df, horizon=horizon, n_windows=n_windows)
    all_results = []
    
    for w in windows:
        forecast_df = forecast_fn(w['train'], w['val'])
        result      = evaluate_forecast(forecast_df, w['val'], w['window'])
        all_results.append(result)
    
    all_results = pd.concat(all_results, ignore_index=True)
    
    # Average metrics across windows
    comparison = (
        all_results
        .groupby('model')[['wape', 'rmse', 'mape', 'bias']]
        .mean()
        .round(4)
        .sort_values('wape')  # rank by primary metric
        .reset_index()
    )
    comparison['rank'] = range(1, len(comparison) + 1)
    
    return comparison, all_results

# Run it on naive baseline
comparison_table, detailed = run_backtest(naive_forecast, df)
print("=== Model Comparison (avg across 3 windows) ===")
print(comparison_table.to_string(index=False))

=== Model Comparison (avg across 3 windows) ===
model   wape    rmse   mape  bias  rank
naive 0.2356 15.5386 0.2707  5.45     1


In [51]:
def load_external_forecast(filepath, model_name):
    """
    Loads a forecast CSV from a teammate.
    Expected columns: date, store, item, y_hat, model
    If 'model' column is missing, fills it with model_name.
    """
    fc = pd.read_csv(filepath)
    fc[DATE_COL] = pd.to_datetime(fc[DATE_COL])
    if 'model' not in fc.columns:
        fc['model'] = model_name
    return fc[[DATE_COL, STORE_COL, ITEM_COL, 'y_hat', 'model']]

In [52]:
# Load per-window classical forecasts (prophet + sarima, top 10 series, 3 windows)
classical_bt = load_external_forecast('../data/processed/forecasts_classical_backtest.csv', model_name=None)
baselines_fc  = load_external_forecast('../data/processed/forecasts_baselines.csv', model_name=None)

# Get the top 10 series that classical models cover — all models scored on same series
top10_keys = classical_bt[[STORE_COL, ITEM_COL]].drop_duplicates()

def combined_forecast(train_df, val_df):
    val_start = val_df[DATE_COL].min()
    val_end   = val_df[DATE_COL].max()

    # Classical forecasts for this window
    classical_window = classical_bt[
        (classical_bt[DATE_COL] >= val_start) &
        (classical_bt[DATE_COL] <= val_end)
    ].copy()

    # Baseline forecasts filtered to same 10 series and same window
    baselines_window = baselines_fc[
        (baselines_fc[DATE_COL] >= val_start) &
        (baselines_fc[DATE_COL] <= val_end)
    ].merge(top10_keys, on=[STORE_COL, ITEM_COL])

    return pd.concat([classical_window, baselines_window], ignore_index=True)

comparison_table, detailed = run_backtest(combined_forecast, df)
print("=== Model Comparison (same 10 series, avg across 3 windows) ===")
print(comparison_table.to_string(index=False))

=== Model Comparison (same 10 series, avg across 3 windows) ===
           model   wape    rmse   mape    bias  rank
         prophet 0.0892 12.2081 0.0938  1.6336     1
          sarima 0.1153 15.7683 0.1214  4.3679     2
            ma_7 0.2066 22.8457 0.2370 16.5107     3
           naive 0.2073 23.1153 0.2337 14.7250     4
seasonal_naive_7 0.2428 26.5541 0.2690 16.5107     5
           ma_28 0.3528 35.4552 0.3935 31.9429     6


In [53]:
import os
os.makedirs('../data/processed', exist_ok=True)

comparison_table.to_csv('../data/processed/model_comparison_shryes.csv', index=False)
detailed.to_csv('../data/processed/backtest_detailed_shryes.csv', index=False)

print("Saved:")
print("  data/processed/model_comparison_shryes.csv")
print("  data/processed/backtest_detailed_shryes.csv")
print(comparison_table)

Saved:
  data/processed/model_comparison_shryes.csv
  data/processed/backtest_detailed_shryes.csv
              model    wape     rmse    mape     bias  rank
0           prophet  0.0892  12.2081  0.0938   1.6336     1
1            sarima  0.1153  15.7683  0.1214   4.3679     2
2              ma_7  0.2066  22.8457  0.2370  16.5107     3
3             naive  0.2073  23.1153  0.2337  14.7250     4
4  seasonal_naive_7  0.2428  26.5541  0.2690  16.5107     5
5             ma_28  0.3528  35.4552  0.3935  31.9429     6
